# Demographic Table — Full Hartwig Cohort (Supplementary Table S3)

Produces **Supplementary Table S3**: the full pan-cancer Hartwig WGS cohort's
patient and tumour characteristics, stratified by ICI-treatment status. This
is the discovery/validation cohort behind the CIDER Score landscape and
technical validation (Figure 2) — distinct from the ICI-treated subset with
known RECIST response used for the clinical response/survival analyses
(**Table 1**, main text; see `11_demographic_table.ipynb`).

Continuous variables: median (IQR); Wilcoxon rank-sum p-value.  
Categorical variables: n (%); χ² p-value (Fisher's exact for small cells).

**Input:** `ready.rds`  
**Output:** `figs/table_s3_demographics_full_cohort.csv` -- kept purely as the
intermediate `assemble_supplementary_tables_excel.R` reads to build the combined
`supplementary_tables.xlsx`; no standalone `.html`/`.docx` are generated for this
table (see the combined workbook instead).


In [ ]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))  # tidyverse + fct_pin_last() + full_cohort_name()
source(paste0(dirname(getwd()), "/helpers/config.r"))

dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)

library(gtsummary)


In [ ]:
# ── Load full-cohort data ──────────────────────────────────────────────────────
# ready.rds stores one row per (sampleId, gp) across four gp levels
# (T-cell overall + its 3 loci), *and* a duplicated
# copy of every row with cohort_broad overwritten to "ICI-Treated" (added in
# 4_base.ipynb so Figure 3 can facet pooled-vs-per-cancer-type together).
# Restricting to gp == "T-cell" removes the first duplication; distinct()
# removes the second. Without both, every patient is counted twice.
# This matches the population used for the Figure 2 landscape validation.
df_raw <- readRDS(paste0(CIDER_PROJ_DIR, "ready.rds")) |>
  filter(gp == "T-cell") |>
  distinct(sampleId, .keep_all = TRUE)


In [ ]:
# ── Variable preparation ──────────────────────────────────────────────────────
# tmbPerMb is the raw value; no back-transform needed for display.
# AmberMeanDepth used as WGS sequencing depth proxy.
# cohort_pan is the 22-level pan-database cohort scheme (Figure 2A), distinct
# from cohort_broad's 5-group ICI-focused scheme used in Table 1. Labelled
# "Cohort" here (not "ICI Cohort" as in Table 1), since this covers every
# patient in this table, ICI-treated or not. Shown as full tumor-type names
# rather than bare acronyms (full_cohort_name(), helpers/shortcuts.r), since
# this is a supplemental table rather than a space-constrained figure axis.
#
# Variable order: patient demographics (Age, Sex), then Prior treatment and
# a data-completeness summary row (Treatment, response & survival data
# available), then the tumour/genomic block — Purity, WGS depth, CIDER
# T-cell score, TMB, MSI-H — then the cohort grouping variable (Cohort).

demo <- df_raw |>
  mutate(
    `ICI-treated`        = factor(
      ifelse(cpi, "ICI-Treated", "Non-ICI Treated"),
      levels = c("ICI-Treated", "Non-ICI Treated")
    ),
    Age                  = as.numeric(age),
    # sex retains whatever casing/abbreviation clinical_go.txt uses
    # (5_models.ipynb defensively matches both "female" and "f" via
    # tolower(), suggesting the raw values aren't a single consistent
    # form). Normalize case-insensitively and output clean, capitalized
    # labels rather than passing the raw column through unchanged.
    Sex                  = factor(
      case_when(
        tolower(sex) %in% c("female", "f") ~ "Female",
        tolower(sex) %in% c("male",   "m") ~ "Male",
        TRUE ~ NA_character_
      ),
      levels = c("Female", "Male")
    ),
    # pretreated in cpi_ready.rds/ready.rds takes numeric values 1/0 (not
    # text) — confirmed directly against the data. == coerces correctly
    # whether pretreated is stored as numeric or character "1"/"0".
    `Prior treatment`    = factor(
      case_when(
        pretreated == 1 ~ "Yes",
        pretreated == 0 ~ "No",
        TRUE ~ NA_character_
      ),
      levels = c("Yes", "No")
    ),
    # Data-completeness summary: what fraction of patients in each column
    # have a recorded treatment regimen, a RECIST response call, AND
    # complete overall-survival follow-up (osDays + osEvent) — the same
    # three-field requirement 4_base.ipynb uses to build cpi_ready.rds
    # (drop_na(response, osDays, osEvent)). treatment/response/OS are
    # populated mainly for ICI-treated patients in this dataset, so this
    # row is expected to differ sharply between the ICI-treated and
    # untreated columns, and its "Yes" count among ICI-treated patients
    # should land close to Table 1's N (the gap being patients with cpi
    # == TRUE but incomplete outcome data).
    `Treatment, response & survival data available` = factor(
      ifelse(
        !is.na(treatment) & !is.na(response) & !is.na(osDays) & !is.na(osEvent),
        "Yes", "No"
      ),
      levels = c("Yes", "No")
    ),
    `Purity (%)`         = as.numeric(purity) * 100,  # stored as 0-1 fraction; convert for display
    `WGS depth`          = as.numeric(AmberMeanDepth),
    `CIDER T-cell score` = as.numeric(cider_score_depth_adj),
    `TMB (mut/Mb)`       = tmbPerMb,
    `MSI-H`              = factor(
      ifelse(msi_high == 1, "Yes", "No"),
      levels = c("Yes", "No")
    ),
    # Full tumor-type names (not bare acronyms) for this supplemental
    # table -- see full_cohort_name() in helpers/shortcuts.r.
    `Cohort`             = factor(
      full_cohort_name(cohort_pan),
      levels = full_cohort_name(PAN_LEVELS)
    )
  ) |>
  select(
    `ICI-treated`,
    Age,
    Sex,
    `Prior treatment`,
    `Treatment, response & survival data available`,
    `Purity (%)`,
    `WGS depth`,
    `CIDER T-cell score`,
    `TMB (mut/Mb)`,
    `MSI-H`,
    `Cohort`
  )


In [ ]:
# ── Build Table S3 ─────────────────────────────────────────────────────────────
# Continuous: median (IQR), Wilcoxon rank-sum.
# Categorical: n (%), χ² (Fisher's exact for small cells).
# missing = "ifany" (unlike Table 1's "no"): surface an "Unknown" row for
# any variable that actually has missing values, rather than silently
# dropping them from the denominator — this is the full pan-cancer cohort,
# where missingness patterns (e.g. MSI-H, Cohort) are worth seeing rather
# than hiding. Age, Sex, and WGS depth are the exception: their "Unknown"
# rows are dropped below via remove_row_type(), since missingness there
# isn't informative for this table's purpose and just adds clutter.

tbl <- demo |>
  tbl_summary(
    by        = `ICI-treated`,
    # "Treatment, response & survival data available" is a Yes/No
    # completeness flag; showing only the "Yes" row (dichotomous) gives a
    # single line ("N (%) with data") instead of a redundant Yes/No pair.
    type      = list(`Treatment, response & survival data available` ~ "dichotomous"),
    value     = list(`Treatment, response & survival data available` ~ "Yes"),
    statistic = list(
      all_continuous()  ~ "{median} ({p25}–{p75})",
      all_categorical() ~ "{n} ({p}%)"
    ),
    digits    = list(
      all_continuous()  ~ 0,
      # TMB keeps 1 decimal (unlike other continuous variables): the paper
      # uses TMB_HIGH_THRESHOLD = 10 (helpers/config.r) as a clinical
      # cutoff, and rounding to whole numbers would blur values right next
      # to that threshold (e.g. 9.6 -> 10). This override is listed after
      # the all_continuous() rule above, so it takes precedence for this
      # one variable.
      `TMB (mut/Mb)`    ~ 1,
      all_categorical() ~ c(0, 0)
    ),
    missing   = "ifany"
  ) |>
  add_overall() |>
  add_p(
    test = list(
      all_continuous()  ~ "wilcox.test",
      all_categorical() ~ "chisq.test.no.correct"
    ),
    pvalue_fun = ~style_pvalue(.x, digits = 3)
  ) |>
  # Drop the "Unknown" missingness row for Age, Sex, and WGS depth only —
  # missing = "ifany" above is still needed globally so it keeps surfacing
  # missingness for the other variables (e.g. MSI-H, Cohort).
  remove_row_type(variables = c(Age, Sex, `WGS depth`), type = "missing") |>
  # Show N and % of the total in each comparison column's header (but not
  # the Overall column, whose % would always just read 100%), so the split
  # between ICI-treated and untreated patients is visible at a glance
  # without a reader doing the math.
  modify_header(
    all_stat_cols(stat_0 = FALSE) ~ "**{level}**, N = {n} ({style_percent(p, digits = 1)}%)"
  ) |>
  modify_caption("**Supplementary Table S3. Full Hartwig cohort characteristics**") |>
  bold_labels()

tbl


In [ ]:
# ── Save outputs ──────────────────────────────────────────────────────────────
out_dir <- paste0(CIDER_PROJ_DIR, "figs/")

# CSV — build intermediate only. Not a standalone deliverable: this
# table's actual output is its sheet in the combined
# supplementary_tables.xlsx (assemble_supplementary_tables_excel.R reads
# this .csv + the metadata sidecar below to build it). strip_gt_markdown()
# (helpers/shortcuts.r) removes the literal "**"/"__" markdown gtsummary
# uses for header/label bold formatting, which a plain CSV can't render --
# left in, it would leak into the combined workbook as literal
# asterisks/underscores around column and row names.
tbl |>
  as_tibble() |>
  dplyr::rename_with(strip_gt_markdown) |>
  dplyr::mutate(dplyr::across(dplyr::where(is.character), strip_gt_markdown)) |>
  readr::write_csv(paste0(out_dir, "table_s3_demographics_full_cohort.csv"))

message("Saved: table_s3_demographics_full_cohort.csv")

# Metadata sidecar (title + footnote) -- consumed by
# assemble_supplementary_tables_excel.R to build the combined supplemental
# tables Excel workbook without re-deriving this table's title/footnote
# text. footnote is empty for this table (none is currently shown).
jsonlite::write_json(
  list(
    title    = "Supplementary Table S3. Full Hartwig cohort characteristics",
    footnote = ""
  ),
  paste0(out_dir, "table_s3_demographics_full_cohort_meta.json"),
  auto_unbox = TRUE
)